In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [34]:
df=pd.read_csv("../../data/raw/major_leagues v1/sofascore_major_leagues_2526season.csv")
df_yearly=pd.read_csv("../../data/raw/major_leagues v1/sofascore_MLS_Argentina_26season.csv")

In [35]:
df.shape

(8056, 117)

In [36]:
df_yearly.shape

(1539, 117)

In [37]:
set(df.columns) == set(df_yearly.columns)

True

In [38]:
df.columns.tolist() == df_yearly.columns.tolist()

True

In [39]:
total_df = pd.concat([df, df_yearly], ignore_index=True)

In [40]:
total_df.shape

(9595, 117)

In [41]:
total_df = total_df.rename(columns={col: f"{col}".lower() for col in total_df.columns})

In [42]:
total_df=total_df.drop(columns=["outfielderblocks","totwappearances","goalsprevented"])

In [43]:
total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]

C:\Users\vibha\AppData\Local\Temp\ipykernel_22124\2912888129.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
C:\Users\vibha\AppData\Local\Temp\ipykernel_22124\2912888129.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]


In [44]:
per90_cols = [
    # Finishing
    "goals",
    "expectedgoals",
    "goals_minus_xg",
    "totalshots",
    "shotsontarget",
    "shotsofftarget",
    "shotsfrominsidethebox",
    "shotsfromoutsidethebox",
    "hitwoodwork",
    "leftfootgoals",
    "rightfootgoals",
    "headedgoals",
    "goalsfrominsidethebox",
    "goalsfromoutsidethebox",
    "freekickgoal",
    "penaltiestaken",
    "penaltygoals",
    "bigchancesmissed",

    # Creativity
    "assists",
    "expectedassists",
    "assists_minus_xa",
    "goalsassistssum",
    "totalpasses",
    "accuratepasses",
    "inaccuratepasses",
    "totaloppositionhalfpasses",
    "accurateoppositionhalfpasses",
    "totalownhalfpasses",
    "accurateownhalfpasses",
    "accuratefinalthirdpasses",
    "keypasses",
    "totalattemptassist",
    "passtoassist",
    "bigchancescreated",
    "totallongballs",
    "accuratelongballs",
    "totalchippedpasses",
    "accuratechippedpasses",
    "totalcross",
    "accuratecrosses",

    # Possession
    "touches",
    "possessionlost",
    "dispossessed",
    "totalcontest",
    "successfuldribbles",
    "offsides",
    "wasfouled",
    "fouls",

    # Defensive
    "tackles",
    "tackleswon",
    "interceptions",
    "ballrecovery",
    "possessionwonattthird",
    "clearances",
    "blockedshots",
    "dribbledpast",
    "errorleadtoshot",
    "errorleadtogoal",

    # Duels
    "totalduelswon",
    "duellost",
    "groundduelswon",
    "aerialduelswon",
    "aeriallost",
    
    # Goalkeeping
    "saves",
    "savescaught",
    "savesparried",
    "savedshotsfrominsidethebox",
    "savedshotsfromoutsidethebox",
    "runsout",
    "successfulrunsout",
    "goalkicks",
    "punches",
    "highclaims",
    "crossesnotclaimed",  
    

    # Discipline
    "yellowcards",
    "yellowredcards",
    "directredcards",
    "owngoals"
]

rate_cols = [
    "goalconversionpercentage",
    "scoringfrequency",

    "accuratepassespercentage",
    "accuratelongballspercentage",
    "successfuldribblespercentage",

    "tackleswonpercentage",

    "totalduelswonpercentage",
    "groundduelswonpercentage",
    "aerialduelswonpercentage",

    "penaltyconversion",

    "rating",
    "totalrating",
    "countrating"
]

for col in per90_cols:
    total_df[f"{col}_per90"] = np.where(
        total_df["minutesplayed"] > 0,
        (total_df[col] / total_df["minutesplayed"]) * 90,
        np.nan
    )

total_df["goals_per_xg"] = np.where(
    total_df["expectedgoals"] > 0,
    total_df["goals"] / total_df["expectedgoals"],
    1
)

total_df["shots_on_target_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsontarget"] / total_df["totalshots"],
    0
)


total_df["inside_box_shot_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsfrominsidethebox"] / total_df["totalshots"],
    0
)

total_df["assist_conversion"] = np.where(
    total_df["keypasses"] > 0,
    total_df["assists"] / total_df["keypasses"],
    0
)


total_df["xa_per_keypass"] = np.where(
    total_df["keypasses"] > 0,
    total_df["expectedassists"] / total_df["keypasses"],
    0
)


total_df["final_third_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accuratefinalthirdpasses"] / total_df["accuratepasses"],
    0
)

total_df["opp_half_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accurateoppositionhalfpasses"] / total_df["accuratepasses"],
    0
)

total_df["dribbles_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["successfuldribbles"] / total_df["touches"],
    0
)

total_df["dispossessed_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["dispossessed"] / total_df["touches"],
    0
)

total_df["possession_lost_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["possessionlost"] / total_df["touches"],
    0
)

total_df["defensive_actions"] = (
    total_df["tackles"] +
    total_df["interceptions"] +
    total_df["ballrecovery"]
)

total_df["defensive_actions_per90"] = (
    total_df["tackles_per90"] +
    total_df["interceptions_per90"] +
    total_df["ballrecovery_per90"]
)

C:\Users\vibha\AppData\Local\Temp\ipykernel_22124\3909454853.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_per90"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_22124\3909454853.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_per_xg"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_22124\3909454853.py:130: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

In [45]:
total_df=total_df[total_df['minutesplayed'] > 900]

In [46]:
total_df['league'].value_counts()

league
England EFL Championship      436
Spain La Liga 2               387
Italy Serie B                 350
Spain La Liga                 347
England Premier League        341
Italy Serie A                 339
Turkiye Super Lig             289
France Ligue 2                288
Germany Bundesliga            285
Portugal Primeira Liga        282
France Ligue 1                277
Germany 2.Bundesliga          276
Saudi Arabia Pro League       270
Netherlands Eredivisie        268
Argentina Liga Profesional    239
USA MLS                       214
Name: count, dtype: int64

In [47]:
total_df.shape

(4888, 206)

In [48]:
metadata_features = [
    "player",
    "team",
    "player id",
    "team id",
    "league",
    "position",
    "minutesplayed",
    "matchesstarted",
    "appearances"
]

In [49]:
total_df=total_df.fillna(0.0)

In [50]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    null_values=total_df.isna().sum().sum()
    print(null_values)

0


In [51]:
total_rows = len(total_df)
unique_ids = total_df['player id'].nunique()
duplicate_count = total_rows - unique_ids
duplicate_count

16

In [52]:
features_to_scale = [
    col for col in total_df.columns 
    if total_df[col].dtype in [np.float64, np.int64] 
    and col not in metadata_features
]

for col in features_to_scale:
    grouped = total_df.groupby(["league", "position"])[col]
    group_mean = grouped.transform("mean")
    group_std = grouped.transform("std")
    
    total_df[f"{col}_zscore"] = np.where(
        group_std > 0,
        (total_df[col] - group_mean) / group_std,
        0.0
    )

C:\Users\vibha\AppData\Local\Temp\ipykernel_22124\1788466843.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_zscore"] = np.where(


In [53]:
total_df.columns.to_list()

['accuratechippedpasses',
 'accuratecrosses',
 'accuratecrossespercentage',
 'accuratefinalthirdpasses',
 'accuratelongballs',
 'accuratelongballspercentage',
 'accurateoppositionhalfpasses',
 'accurateownhalfpasses',
 'accuratepasses',
 'accuratepassespercentage',
 'aerialduelswon',
 'aerialduelswonpercentage',
 'aeriallost',
 'appearances',
 'assists',
 'attemptpenaltymiss',
 'attemptpenaltypost',
 'attemptpenaltytarget',
 'ballrecovery',
 'bigchancescreated',
 'bigchancesmissed',
 'blockedshots',
 'cleansheet',
 'clearances',
 'countrating',
 'crossesnotclaimed',
 'directredcards',
 'dispossessed',
 'dribbledpast',
 'duellost',
 'errorleadtogoal',
 'errorleadtoshot',
 'expectedassists',
 'expectedgoals',
 'fouls',
 'freekickgoal',
 'goalconversionpercentage',
 'goalkicks',
 'goals',
 'goalsassistssum',
 'goalsconceded',
 'goalsconcededinsidethebox',
 'goalsconcededoutsidethebox',
 'goalsfrominsidethebox',
 'goalsfromoutsidethebox',
 'groundduelswon',
 'groundduelswonpercentage',
 'h

In [54]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(total_df[total_df.duplicated(subset=['player id'], keep=False)].drop_duplicates(subset=['player id'])[['player id', 'player']])

,player id,player
101,1013842,Emmanuel Agbadou
491,361352,Adam Armstrong
688,892521,Unai Núñez
818,1084399,Carlos Vicente
904,1513222,Ángel Pérez
1878,852404,Mattéo Guendouzi
1919,996820,Kristjan Asllani
1928,959806,Kenneth Taylor
2350,839493,Amir Murillo
2560,959805,Quinten Timber


In [55]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(total_df[['team', 'team id']].drop_duplicates().dropna())

,team,team id
5,Brentford,50
6,Brighton & Hove Albion,30
7,Leeds United,34
8,Manchester City,17
9,Manchester United,35
10,Everton,48
13,Burnley,6
14,Sunderland,41
15,Aston Villa,40
17,Crystal Palace,7


In [56]:
player_team_ids = {

1013842: 3050, # Emmanuel Agbadou → Beşiktaş (permanent from Wolves, Feb 6 2026)

361352: 3, # Adam Armstrong → Wolverhampton Wanderers (from Southampton, Feb 2 2026)

892521: 2828, # Unai Núñez → Valencia (loan from Celta Vigo, Jan 2026)

1084399: 9, # Carlos Vicente → Birmingham City (from Deportivo Alavés, Feb 2026)

1513222: 2885, # Ángel Pérez → Deportivo Alavés (from Huesca, Jan 28 2026)

852404: 3052, # Mattéo Guendouzi → Fenerbahçe (from Lazio, Jan 8 2026)

996820: 3050, # Kristjan Asllani → Beşiktaş (loan from Inter Milan, Jan 30 2026)

959806: 2699, # Kenneth Taylor → Lazio (from Ajax, Jan 9 2026)

839493: 3050, # Amir Murillo → Beşiktaş (from Marseille, Feb 6 2026)

959805: 1641, # Quinten Timber → Olympique de Marseille

343013: 12, # Ethan Horvath → Sheffield Wednesday (loan from Cardiff City)

1130805: 269199,# Iker Kortajarena → Al-Kholood (from Huesca, Jan/Feb 2026)

1499082: 3036, # Jalen Blesa → Rio Ave (from Cesena, Feb 2 2026)

992024: 190328,# Rhaldney → FC Alverca (loan from Göztepe, Feb 3 2026)

234148: 3052, # N'Golo Kanté → Fenerbahçe (from Al-Ittihad, Feb 3 2026)

846470: 34315, # Youssef En-Nesyri → Al-Ittihad (from Fenerbahçe, Feb 3 2026)

}

In [57]:
for player_id, target_team_id in player_team_ids.items():
    player_rows = total_df[total_df["player id"] == player_id]
    if len(player_rows) <= 1:
        continue
        
    total_mins = player_rows["minutesplayed"].sum()
    if total_mins == 0:
        continue
        
    target_idx = player_rows[player_rows["team id"] == target_team_id].index
    if len(target_idx) == 0:
        target_idx = [player_rows.index[0]]
        
    keep_idx = target_idx[0]
    drop_indices = player_rows.index[player_rows.index != keep_idx]
    
    total_df.loc[keep_idx, "minutesplayed"] = total_mins
    
    for col in total_df.columns:
        if col in metadata_features and col != "minutesplayed":
            continue
            
        if col.endswith("_zscore"):
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
        elif col.endswith("_per90") or col in per90_cols:
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            if total_df[col].dtype in [np.int64, np.int32]:
                total_df = total_df.astype({col: "float64"})
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
        elif col in rate_cols or col in ["goals_per_xg", "shots_on_target_pct", "inside_box_shot_pct", "assist_conversion", "xa_per_keypass", "final_third_pass_pct", "opp_half_pass_pct", "dribbles_per_touch", "dispossessed_per_touch", "possession_lost_per_touch", "weak_foot_goals_pct"]:
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            if total_df[col].dtype in [np.int64, np.int32]:
                total_df = total_df.astype({col: "float64"})
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
            
    total_df = total_df.drop(drop_indices)

total_df = total_df.reset_index(drop=True)

In [64]:
total_df.to_csv("../../data/processed/major_leagues/Sofascore_player_data_2526.csv",index=False)